In [14]:
from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from pathlib import Path
from PIL import Image

#import to cover extras from single_trial.py
import gc
import scipy.io
from scipy import stats
from scipy.signal import butter, lfilter
from scipy.signal import savgol_filter
import sys
import mat73
import pandas as pd


from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir
from caiman.summary_images import local_correlations_movie_in_memory
import gc


logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
%reload_ext autoreload

In [2]:
#os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [2]:
fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM\FOV1_T2.tsm'
fr = 640
print(fname, fr)

C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM\FOV1_T2.tsm 640


In [3]:
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

In [4]:
m_orig = cm.load(fname)
ds_ratio = 0.2


In [6]:

# moviehandle = m_orig.resize(1, 1, ds_ratio)
# moviehandle.play(q_max=99.5, fr=40, magnification=1)

In [5]:
c, dview, n_processes = cm.cluster.setup_cluster(
            backend='local', n_processes=None, single_thread=False)


In [8]:
#  dview.terminate()

In [7]:

mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=False)
#about 2.3 minutes for 12800 frames (2m 13-21 s)

In [8]:

#m_orig = cm.load(fname)
#m_rig = cm.load(mc.mmap_file)
m_rig = mc.apply_shifts_movie(fname) ###This will take a minute or so
ds_ratio = 0.2
#moviehandle = cm.concatenate([m_orig.resize(1, 1, ds_ratio) - mc.min_mov*mc.nonneg_movie,
                              #m_rig.resize(1, 1, ds_ratio)], axis=2)
#moviehandle.play(fr=60, q_max = 99.5, magnification=1)

# 43.7 s

In [10]:
import gc
from pathlib import Path


def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)


# 1. Delete any Python references to memmaps pointing to R:/
try:
    safe_close_mmap(Yr)  # or whatever your memmap object is called
except NameError:
    pass

try:
    safe_close_mmap(mmap_file_rig)  # or whatever your memmap object is called
except NameError:
    pass

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")

Cleared all files from R:/


In [11]:


#border_to_0 = 0 if mc.border_nan == 'copy' else mc.border_to_0
#fname_new = cm.save_memmap_join(mc.mmap_file, base_name='memmap_', add_to_mov=border_to_0, dview=dview)


# Path to RAM disk memmap
p = Path(fname)
ram_path = Path(r'R:/') / f"{p.stem}_rig__d1_{m_rig.shape[1]}_d2_{m_rig.shape[2]}_d3_1_order_C_frames_{m_rig.shape[0]}.mmap"
ram_path = str(ram_path).replace("/", "\\")

# Create memmap in RAM-disk with same shape as m_rig
mmap_file_rig = np.memmap(ram_path, dtype='float32', mode='w+', shape=m_rig.shape, order='F') #Was C before

# Copy stabilized movie data into memmap
mmap_file_rig[:] = m_rig[:]

# Flush to make sure data is written
mmap_file_rig.flush()

mmap_list = [mmap_file_rig]

print("Saved stabilized memmap to RAM-disk:", ram_path)


Saved stabilized memmap to RAM-disk: R:\FOV1_T2_rig__d1_512_d2_512_d3_1_order_C_frames_12800.mmap


In [12]:
#border_to_0 = 0 if mc.border_nan == 'copy' else mc.border_to_0
#fname_new = cm.save_memmap_join(mc.mmap_file, base_name='memmap_', add_to_mov=border_to_0, dview=dview)

#img = mean_image(mc.mmap_file[0], window=1000, dview=dview)



img = np.mean(m_rig, axis=0)
img = (img-np.mean(img))/np.std(img)

gaussian_blur = False
#cn = local_correlations_movie_offline(mc.mmap_file[0], fr=fr, window=fr*4, stride=fr*4, winSize_baseline=fr*2, remove_baseline=True,
#                                      gaussian_blur = gaussian_blur, dview=dview).max(axis=0)

In [ ]:

cn_movie = local_correlations_movie_in_memory(
    m_rig,
    fr=fr,
    window=fr*4,
    stride=fr*4,
    eight_neighbours=True,
    dview=dview
)

cn = cn_movie.max(axis=0)

#2m 22s

Using in-memory version of local_correlations_movie, no parallelization


In [19]:


print("m_rig shape:", m_rig.shape)
print("img shape:", img.shape)
print("cn shape:", cn.shape)
print("cn_movie shape:", cn_movie.shape)
print("difference:", (img.shape[0] - cn.shape[0], img.shape[1] - cn.shape[1]))

m_rig shape: (12800, 512, 512)
img shape: (512, 512)
cn shape: (512, 512)
cn_movie shape: (5, 512, 512)
difference: (0, 0)


In [20]:




img_corr = (cn - np.mean(cn))/np.std(cn)
summary_images = np.stack([img, img, img_corr], axis=0).astype(np.float32)
cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')



In [21]:

plt.imshow(summary_images[0], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)



In [22]:

plt.imshow(summary_images[2], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)
img = summary_images.transpose([1, 2, 0])


print(fname[:-4]+'_corr.tif')
height, width = img.shape[:2]
print(img.shape)


C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM\FOV1_T2_corr.tif
(512, 512, 3)


In [23]:
# --------------------------------------------------------------
# Extract channels like MATLAB
# --------------------------------------------------------------
R = img[:, :, 0]
B = img[:, :, 2]

# --------------------------------------------------------------
# MATLAB-style normalization (mat2gray + uint8)
# --------------------------------------------------------------
def normalize_like_matlab(x):
    x = x.astype(np.float64)
    mn = x.min()
    mx = x.max()
    x = (x - mn) / (mx - mn + 1e-12)

    # MATLAB uint8 applies rounding, not floor
    x = np.round(255 * x).astype(np.uint8)
    return x

R_norm = normalize_like_matlab(R)
B_norm = normalize_like_matlab(B)

# --------------------------------------------------------------
# Build MATLAB-equivalent RGB (R,R,B)
# --------------------------------------------------------------
rgb = np.stack([R_norm, R_norm, B_norm], axis=2).astype(np.uint8)

# --------------------------------------------------------------
# Save as PNG (MATLAB-compatible pixel data)
# --------------------------------------------------------------
outname = fname[:-4] + "_py.png"
Image.fromarray(rgb).save(outname)

print("Saved:", outname)


Saved: C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM\FOV1_T2_py.png


In [24]:
# #load in my own FOV images
# import imageio.v3 as iio
# import numpy as np

# # Load PNG as RGB (H, W, 3)
# img = iio.imread("C:/Users/ICNLab/caiman_data/testdata/testdata/FOV9_T2-001_py.png")

# #.astype(np.float32)
img = rgb.copy()


In [25]:
print("bob")

bob


In [26]:
weights_path="C:/Users/ICNLab/caiman_data/testdata/testdata/mask_rcnn_neuron_0012.h5"
#download_model('mask_rcnn')
#ROIs, r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
ROIs = r['masks'].transpose([2, 0, 1])
cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')


Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_class_loss': 1.0, 'mrcnn_bbox_loss': 1.0, 'mrcnn_mask_loss': 1.0}
MASK_POOL_SIZE                 14
MASK_SHAPE                  

      871431 [deprecation.py:            new_func():554][27464] From c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\tensorflow\python\util\deprecation.py:629: calling map_fn_v2 (from tensorflow.python.ops.map_fn) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Use fn_output_signature instead


Processing 1 images
image                    shape: (512, 512, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min:  -91.11000  max:  168.24000  float64
image_metas              shape: (1, 14)               min:    0.00000  max:  512.00000  int32
anchors                  shape: (1, 65472, 4)         min:   -0.04428  max:    1.01297  float32


c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


MADE FIGURE


In [27]:
fig, axs = plt.subplots(1, 2)
axs[0].imshow(summary_images[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')
plt.savefig(fname[:-6] + 'newmrcnn_ROIs.png', format='png', bbox_inches='tight', pad_inches=0)

In [28]:
cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)

In [29]:
ROIs = ROIs                                   # region of interests
index = list(range(len(ROIs)))                # index of neurons
weights = None                                # if None, use ROIs for initialization; to reuse weights check reuse weights block
print(str(len(index))+" ROIs")
template_size = 0.02                          # half size of the window length for spike templates, default is 20 ms
context_size = 35                             # number of pixels surrounding the ROI to censor from the background PCA
visualize_ROI = False                         # whether to visualize the region of interest inside the context region
hp_freq_pb = 1 / 3                            # parameter for high-pass filter to remove photobleaching
clip = 100                                    # maximum number of spikes to form spike template
threshold_method = 'adaptive_threshold'       # adaptive_threshold or simple
min_spikes= 10                                # minimal spikes to be found
pnorm = 0.5                                   # a variable deciding the amount of spikes chosen for adaptive threshold method
threshold = 2                                 # threshold for finding spikes only used in simple threshold method, Increase the threshold to find less spikes
do_plot = False                               # plot detail of spikes, template for the last iteration
ridge_bg= 0.05                                # ridge regression regularizer strength for background removement, larger value specifies stronger regularization
sub_freq = 20                                 # frequency for subthreshold extraction
weight_update = 'ridge'                       # ridge or NMF for weight update
n_iter = 2                                    # number of iterations alternating between estimating spike times and spatial filters

opts_dict={'fnames': ram_path, #mmap_file_rig,   #'fnames': fname_new,
           'ROIs': ROIs,
           'index': index,
           'weights': weights,
           'template_size': template_size,
           'context_size': context_size,
           'visualize_ROI': visualize_ROI,
           'hp_freq_pb': hp_freq_pb,
           'clip': clip,
           'threshold_method': threshold_method,
           'min_spikes':min_spikes,
           'pnorm': pnorm,
           'threshold': threshold,
           'do_plot':do_plot,
           'ridge_bg':ridge_bg,
           'sub_freq': sub_freq,
           'weight_update': weight_update,
           'n_iter': n_iter}

opts.change_params(params_dict=opts_dict);

67 ROIs


In [30]:
vpy = VOLPY(n_processes=n_processes, dview=dview, params=opts)


In [ ]:
vpy.fit(n_processes=n_processes, dview=dview)
#takes a while to run
#1m 19s

Starting VOLPY spike detection...


In [32]:
# Visualize spatial footprints and traces
print(np.where(vpy.estimates['locality'])[0])    # neurons that pass locality test
idx = np.where(vpy.estimates['locality'] > 0)[0]
utils.view_components(vpy.estimates, img_corr, idx)

[ 0  1  3  4  5  6  7  8  9 12 13 15 17 18 19 20 21 22 23 24 26 28 29 31
 32 36 37 38 39 41 43 44 46 47 54 55 57 58 59 62 64]
Component:0


In [ ]:
# Reconstructed movie
#flip_signal = True    
#mv_all = utils.reconstructed_movie(vpy.estimates.copy(), fnames=ram_path,           #mc.mmap_file,
#                                           idx=idx, scope=(0,1000), flip_signal=flip_signal)
#mv_all.play(fr=40, magnification=3)

In [ ]:
#print(type(mv_all))
#print(mv_all.size)

<class 'caiman.base.movies.movie'>
786432000


In [ ]:
#mv_uint8 = (mv_all / mv_all.max() * 255).astype(np.uint8)


In [ ]:
#mv_uint8.play(fr=40, magnification=3)


#mv_all.play(fr=40, magnification=3)

In [33]:
vpy.estimates['ROIs'] = ROIs
save_name = fname[:-4]+'new_volpy'
np.save(save_name, vpy.estimates)

cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)

In [34]:
cm.stop_server(dview=dview)

In [35]:
vpy = vpy.estimates

In [36]:
print(vpy)

{'rawROI': array([{'t': array([4.95675373, 6.3902483 , 4.65778255, ..., 2.66629982, 1.44387817,
              1.74194646]), 'ts': array([ 3.6354795 ,  4.4881034 ,  3.4299126 , ...,  0.14423095,
              -0.9282305 , -0.48289013], dtype=float32), 'spikes': array([    8,    12,   232,   254,   379,   388,   401,   405,   708,
                716,   977,  1153,  1156,  1162,  1165,  1169,  1171,  1173,
               1363,  1366,  1498,  1658,  1661,  1667,  1772,  2076,  2204,
               2359,  2362,  2613,  2709,  2801,  2807,  3083,  3087,  3168,
               3174,  3205,  3208,  3211,  3215,  3669,  3891,  4428,  4871,
               5712,  6073,  6382,  6938,  7302,  7410,  7824,  9055,  9180,
               9492,  9495, 10051, 10053, 10057, 10311, 10758, 11268, 11374,
              11656, 11662, 11697, 11937, 12224, 12229, 12235, 12551, 12747,
              12750], dtype=int64), 'weights': array([[False, False, False, ..., False, False, False],
              [False, False

In [37]:
num_frames = np.max(vpy['dFF'].shape)
dur = num_frames/640
vpy['cellID'] = []
vpy['raster'] = np.zeros_like(vpy['dFF'])
vpy['firing_rate'] = np.zeros_like(vpy['dFF'])

for i in range(vpy['dFF'].shape[0]-1):
    vpy['raster'][i,vpy['spikes'][i]] = 1
    vpy['firing_rate'][i] = savgol_filter(np.convolve(vpy['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

    if np.sqrt(np.var(vpy['templates'][i], ddof=1))>0.5:
        vpy['cellID'].append(i)

if len(vpy['cellID'])>0:
    dFF = np.array(vpy['dFF']).astype(float)
    R = np.corrcoef(dFF)
    r = np.array(np.where(np.triu(R,1)>0.7))
    for i in range(0,r.shape[1]):
        if np.max(dFF[r[0][i]]) < np.max(dFF[r[1][i]]):
            r[1][i] = r[0][i]        
    vpy['cellID'] = [x for x in vpy['cellID'] if x not in r[1]]            
    
cells = np.array(vpy['cellID'])

time = np.arange(0,dur,1/640)


In [38]:
fig = plt.figure(figsize=(8.0, 11.0), facecolor='w',constrained_layout=True)
spec = fig.add_gridspec(ncols=3, nrows=5, width_ratios=[1,1,1], height_ratios=[2, 5,1,1,1])
ax1 = fig.add_subplot(spec[0, 0])
ax2 = fig.add_subplot(spec[0, 1])
ax_text = fig.add_subplot(spec[0, 2],facecolor='w')
ax3 = fig.add_subplot(spec[1, :],facecolor='w')
ax4 = fig.add_subplot(spec[4, :],facecolor='w')
ax5 = fig.add_subplot(spec[2, :],facecolor='w')
ax5r = ax5.twinx()
ax6 = fig.add_subplot(spec[3, :],facecolor='w')
ax7 = fig.add_subplot(spec[4, :],facecolor='w')

ax1.imshow(img[:,:,1], cmap='gray')
ax2.imshow(img[:,:,2], cmap='gray')
ax1.set_title('Mean image',color='k',fontsize=14)
ax2.set_title('Corr image',color='k',fontsize=14)
ax1.set_axis_off()
ax2.set_axis_off()
ax_text.set_axis_off()
wheel_mat = os.path.dirname(fname) + '\\Wheel.mat'
if os.path.exists(wheel_mat):    
    wheel=mat73.loadmat(wheel_mat)     
    if 'behavior' in wheel:
        ax4.plot(wheel['behavior'][:,0],wheel['behavior'][:,1],'r',linewidth=1.2)
        if wheel['behavior'].shape[1]>2:
            ax4.plot(wheel['behavior'][:,0],wheel['behavior'][:,2],'k',linewidth=1)
        ax4.set_ylabel('Behavior',color='k',fontsize=12)
        ax4.set_yticks([-1,0,1])
        ax4.set_ylim([-2,2])

    if wheel['data_time'].any():
        whl_time = np.arange(0,np.max(wheel['data_time']),1/640)
        wheel_interp = np.interp(whl_time, wheel['data_time'], wheel['data_pos'])  
        speed = np.zeros_like(wheel_interp)
        for i in range(0,len(whl_time)-1):
            speed[i] = (wheel_interp[i+1]-wheel_interp[i])/(whl_time[i+1]-whl_time[i])

        #speed[speed>100] = 0
        #speed[speed<0] = 0
        speed = savgol_filter(speed,64,1)
        ax6.plot(whl_time,speed,'k',linewidth=1)
        ax6.set_ylabel('Speed (cm/s)',color='k',fontsize=12)
        #ax7.plot(whl_time,speed,'w',linewidth=1)
        #ax7.set_ylabel('Speed (cm/s)',color='k',fontsize=12)
    
    ax_text.text(0.5, 0.8, 'Mouse = ' + wheel['mouse'], color='k',fontsize=10, ha='center')
    ax_text.text(0.5, 0.4, 'Stimulus = ' + wheel['stimulus'], color='k',fontsize=10, ha='center')
    ax_text.text(0.5, 0.6, 'Date = ' + str(np.array(wheel['currentdate'],dtype='int32')), color='k',fontsize=10, ha='center')
    ax_text.text(0.5, 0.2, 'File = ' + wheel['file'], color='w',fontsize=10, ha='center')
    if wheel['stimulus']=='Map' and 'rand_num' in wheel:
        ax_text.text(0.5, 0, 'Field = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
    elif wheel['stimulus']=='Tuning' and 'rand_num' in wheel:
        ax_text.text(0.5, 0, 'Orientation = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
    elif wheel['stimulus']=='Tuning' and 'rand_num' in wheel:
        ax_text.text(0.5, 0, 'Orientation = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')    
else:
    print("Wheel data does not exist")
llim = 0

b, a = butter(1, [1.5, 100], fs=640, btype='band')
k = 1
pos_cells = []
neg_cells = []
for i in range(0, len(cells)):
    if ''.join(vpy['polarity'][cells[i]]) in 'negative':
        color = '#9AAB3A'
        mult = -1
        neg_cells.append(cells[i])
    else:
        color = '#54A0A8'
        mult = 1
        pos_cells.append(cells[i])
    y = np.array(lfilter(b,a,stats.zscore(np.array(vpy['dFF'][cells[i]] * mult * 100,dtype=np.float32))) + ((k - 1) * 8)).reshape(1,num_frames)
    ax3.plot(llim+time,y[0,:],color, linewidth=0.3)
    ax3.plot(llim+time[vpy['spikes'][cells[i]]],np.max(y)*np.ones(vpy['spikes'][cells[i]].shape[0]),"|",color='firebrick',markersize=2)
    k = k + 1

    
if len(pos_cells)>0:
    mean_fr_pos = np.mean(vpy['firing_rate'][pos_cells,:], axis=0)
    sem_pos = stats.sem(np.array(vpy['firing_rate'][pos_cells,:],dtype=np.float32), axis=0)
    ax5r.plot(llim+time, np.array(mean_fr_pos,dtype='float32').ravel(), label='Mean firing rate', color='#54A0A8',linewidth=0.3)
    ax5r.fill_between(llim+time, np.array(mean_fr_pos - sem_pos,dtype='float32').ravel(), np.array(mean_fr_pos + sem_pos,dtype='float32'), color='#54A0A8', alpha=0.3, label='SEM')
    ax5.set_ylabel('Firing rate (Hz)',color='#54A0A8',fontsize=12)
    ax5r.tick_params(axis ='y', labelcolor = '#54A0A8') 
if len(neg_cells)>0:
    mean_fr_neg = np.mean(vpy['firing_rate'][neg_cells,:], axis=0)
    sem_neg = stats.sem(np.array(vpy['firing_rate'][neg_cells,:],dtype=np.float32), axis=0)
    ax5.plot(llim+time, np.array(mean_fr_neg,dtype='float32').ravel(), label='Mean firing rate', color='#9AAB3A',linewidth=0.3)
    ax5.fill_between(llim+time, np.array(mean_fr_neg - sem_neg,dtype='float32').ravel(), np.array(mean_fr_neg + sem_neg,dtype='float32'), color='#9AAB3A', alpha=0.3, label='SEM')
    ax5.set_ylabel('Firing rate (Hz)',color='#9AAB3A',fontsize=12)
    ax5r.tick_params(axis ='y', labelcolor = '#9AAB3A')
    
for ax in [ax3,ax4,ax5,ax6,ax7]:
    ax.tick_params(color='black', labelcolor='black')
    ax.set_xlabel('Time (sec)',color='k',fontsize=12)
    ax.set_xlim([llim,llim+dur])
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
ax3.set_title('dFF',color='k',fontsize=14)
ax3.set_ylabel(r'$\Delta$F/F (%)',color='k',fontsize=12)
pos_cells = []
neg_cells = []

In [40]:
fig.savefig(fname[:-6] + '_volpy_ridge.pdf')

Component:0
Component:0
Component:0
Component:0
Component:1
Component:1
Component:1
Component:1
Component:1
Component:2
Component:2
Component:3
Component:4
Component:5
Component:8
Component:11
Component:13
Component:16
Component:17
Component:18
Component:19
Component:20
Component:22
Component:23
Component:24
Component:26
Component:27
Component:28
Component:29
Component:29
Component:29
Component:29
Component:29
Component:29
Component:28
Component:28
Component:28
Component:28
Component:27
Component:26
Component:25
Component:25
Component:24
Component:21
Component:19
Component:17
Component:15
Component:14
Component:13
Component:12
Component:12
Component:11
Component:11
Component:23
Component:29
Component:32
Component:33
Component:34
Component:35
Component:36
Component:36
Component:36
Component:37
Component:37
Component:37
Component:37
Component:37
Component:37
Component:36
Component:36
Component:36
Component:36
Component:35
Component:35
Component:34
Component:34
Component:34
Component:36
C